Building a Conversation Loop

def run_conversation(messages):
    while True:
        response = chat(messages)

        add_assistant_message(messages, response)

        # Pseudo code
        if response isn't asking for a tool:
            break

        tool_result_blocks = run_tools(response)
        add_user_message(messages, tool_result_blocks)
        
    return messages

In [22]:
from helper_chat import add_assistant_message, add_user_message, chat_response, text_from_message
from helper_tools import get_current_datetime, get_current_datetime_schema, add_duration_to_datetime, add_duration_to_datetime_schema, set_reminder, set_reminder_schema

In [23]:
import json

def run_tool(tool_name, tool_input):
    if tool_name == "get_current_datetime":
        return get_current_datetime(**tool_input)
    elif tool_name == "add_duration_to_datetime":
        return add_duration_to_datetime(**tool_input)
    elif tool_name == "set_reminder":
        return set_reminder(**tool_input)

def run_tools(message):
    tool_requests = [
        block for block in message.content if block.type == "tool_use"
    ]

    tool_result_blocks = []

    for tool_request in tool_requests:
        
        try:
            tool_output = run_tool(tool_request.name, tool_request.input)
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": json.dumps(tool_output),
                "is_error": False 
                }
        except Exception as e:
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": f"Error: {e}",
                "is_error": True
                }

        tool_result_blocks.append(tool_result_block)
    
    return tool_result_blocks

In [24]:
def run_conversation(messages):
    while True:
        response = chat_response(messages, tools=[
            get_current_datetime_schema,
            add_duration_to_datetime_schema,
            set_reminder_schema,
            ])
        add_assistant_message(messages,response)
        print(text_from_message(response))

        if response.stop_reason != "tool_use":
            break

        tool_results = run_tools(response)
        add_user_message(messages,tool_results)
        
    return messages

In [25]:
messages = []

add_user_message(
    messages,
    "Set a reminder for my doctors appointment. Its 177 days after Jan 1st 2050."
)

run_conversation(messages)

I'll help you set a reminder for your doctor's appointment. First, let me calculate the date that is 177 days after January 1st, 2050.
Now I'll set a reminder for your doctor's appointment on June 27, 2050:
----
Setting the following reminder for 2050-06-27T00:00:00:
Doctor's appointment
----
Perfect! I've set a reminder for your doctor's appointment on **Monday, June 27, 2050** at 12:00 AM. You'll receive a notification at that time.


[{'role': 'user',
  'content': 'Set a reminder for my doctors appointment. Its 177 days after Jan 1st 2050.'},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text="I'll help you set a reminder for your doctor's appointment. First, let me calculate the date that is 177 days after January 1st, 2050.", type='text'),
   ToolUseBlock(id='toolu_018PNJUX8cYnMZiCwZF1nbrJ', caller=DirectCaller(type='direct'), input={'datetime_str': '2050-01-01', 'duration': 177, 'unit': 'days'}, name='add_duration_to_datetime', type='tool_use')]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_018PNJUX8cYnMZiCwZF1nbrJ',
    'content': '"Monday, June 27, 2050 12:00:00 AM"',
    'is_error': False}]},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text="Now I'll set a reminder for your doctor's appointment on June 27, 2050:", type='text'),
   ToolUseBlock(id='toolu_0125pzuSZ77D9BTUjrhs6a7C', caller=DirectCaller(type='direct'), input={'content': "Do